Basic LangGraph Agent with Two Simple Nodes

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

c:\Users\Arun.Manglick\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
from langchain_ollama import ChatOllama

# Using the available model in your Ollama environment
llm = ChatOllama(model="gpt-oss:120b-cloud")
llm.invoke("Capital of France")

AIMessage(content='The capital of France is **Paris**. It’s also the country’s largest city and a major cultural, political, and economic hub.', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b-cloud', 'created_at': '2026-01-02T13:00:47.404745552Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1086139756, 'load_duration': None, 'prompt_eval_count': 70, 'prompt_eval_duration': None, 'eval_count': 61, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b-cloud', 'model_provider': 'ollama'}, id='lc_run--019b7ecb-d4bf-7fc2-89b9-c8720a49d74a-0', usage_metadata={'input_tokens': 70, 'output_tokens': 61, 'total_tokens': 131})

In [ ]:
from typing import Annotated

class ChatState(TypedDict): 
    chatMessages: Annotated[list, add_messages]

def chatbotMessage(state: ChatState) -> ChatState:
    response = llm.invoke(state["chatMessages"])
    return {"chatMessages": [response]}

Define LangGraph Edges and Nodes

In [ ]:
builder = StateGraph(ChatState)
builder.add_node("chatbotMessage_node", chatbotMessage)

builder.add_edge(START, "chatbotMessage_node")
builder.add_edge("chatbotMessage_node", END)

graph = builder.compile()

Display Graph

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

Example Talk to Chat Agent

In [ ]:
message = {"role": "user", "content": "What is the capital of France?"}
result = graph.invoke({"chatMessages": [message]})
result["chatMessages"]

Talk to Agent

Define Chat Widget using IpWidgets

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create widgets
user_input = widgets.Text(
    placeholder='Type your message here...',
    description='You:',
    layout=widgets.Layout(width='80%')
)
send_button = widgets.Button(
    description='Send',
    button_style='success'
)
quit_button = widgets.Button(
    description='Quit',
    button_style='danger'
)
output_area = widgets.Output()

# Store state
chat_widget_state = {"state": None, "active": True}

def send_message(b):
    if not chat_widget_state["active"]:
        return
        
    message = user_input.value.strip()
    if not message:
        return
    
    with output_area:
        print(f"You: {message}")
        
        # Initialize or update state
        if chat_widget_state["state"] is None:
            chat_widget_state["state"] = {
                "chatMessages": [{"role": "user", "content": message}]
            }
        else:
            chat_widget_state["state"]["chatMessages"].append({"role": "user", "content": message})
        
        # Get bot response
        chat_widget_state["state"] = graph.invoke(chat_widget_state["state"])
        bot_response = chat_widget_state["state"]["chatMessages"][-1].content
        print(f"Bot: {bot_response}\n")
    
    user_input.value = ''  # Clear input

def quit_chat(b):
    chat_widget_state["active"] = False
    with output_area:
        print("\n--- Chat ended ---")
    user_input.disabled = True
    send_button.disabled = True
    quit_button.disabled = True


# Bind events
send_button.on_click(send_message)
quit_button.on_click(quit_chat)
user_input.on_submit(lambda x: send_message(None))  # Allow Enter key

Display UI

In [ ]:
# Display UI
print("Chat Interface (Type your message and press Send or Enter)")
display(widgets.HBox([user_input, send_button, quit_button]))
display(output_area)